# 03 — ResNet50 normalization

This is the exact normalization used by the ResNet50 pipeline. Images are decoded as RGB, resized to 224×224, converted to float tensors, and passed through **ResNet50’s `preprocess_input`**. That function converts RGB to the channel convention and scales values expected by ImageNet-pretrained ResNet50.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input

IMAGE_SIZE=(224,224)

def normalize_for_resnet50(image, label):
    image=tf.image.resize(image, IMAGE_SIZE)
    image=tf.cast(image, tf.float32)
    image=preprocess_input(image)  # actual ResNet50 preprocessing executed
    return image, label

print("Normalization function:", normalize_for_resnet50)


In [ ]:
# Example dataset construction from the saved manifest
from pathlib import Path
import pandas as pd

ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=="03Normalization" else ROOT
manifest=pd.read_csv(ROOT/"results"/"preprocessing"/"split_manifest.csv")
classes=sorted(manifest["class"].unique()); class_to_id={c:i for i,c in enumerate(classes)}

def load_image(path, label):
    image=tf.io.read_file(path)
    image=tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None,None,3])
    return normalize_for_resnet50(image,label)

train_rows=manifest[manifest.split=="train"]
train_ds=tf.data.Dataset.from_tensor_slices((train_rows.path.values, train_rows["class"].map(class_to_id).values))
train_ds=train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
print(train_ds)


**Viva answer:** the notebook executes `preprocess_input(image)` from `tensorflow.keras.applications.resnet50`; it is not simple division by 255. Validation and test images use the same deterministic normalization without random augmentation.